# Qwen3 1.7B on CoreML

In [1]:
import torch
import coremltools as ct
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).eval()

import numpy as np
# Qwen3-1.7B typical dims (Verify with model.config)
# Layers: 24, Heads: 16, Head_Dim: 128 (examples)
num_layers = model.config.num_hidden_layers
num_heads = model.config.num_key_value_heads
head_dim = model.config.hidden_size // model.config.num_attention_heads
max_seq_len = 4096

scikit-learn version 1.7.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.11.0.dev20251222 has not been tested with coremltools. You may run into unexpected errors. Torch 2.7.0 is the most recent version that has been tested.
W0211 17:38:49.607000 83928 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0211 17:38:51.583000 83928 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0211 17:38:51.583000 83928 torch/utils/flop_counter.py:45] triton not found; flop counting will not work for triton kernels
W0211 17:38:51

In [5]:
from coremltools.optimize.torch.quantization import LinearQuantizer, LinearQuantizerConfig, ModuleLinearQuantizerConfig

class StatefulQwen(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, input_ids):
        # The model uses its internal buffers (registered above) 
        # as the 'past_key_values' automatically in recent transformers
        outputs = self.model(input_ids, use_cache=False)
        return outputs.logits

# Define the config for W4A8
# Note: 'activation_dtype' is valid inside ModuleLinearQuantizerConfig
global_config = ModuleLinearQuantizerConfig(
    quantization_scheme="symmetric",
    weight_dtype="qint4", # We will specify n_bits=4 in the next step or during conversion
    activation_dtype=torch.quint8, # Standard for A8
    milestones=[0, 100, 200, 300]
)

model.config.use_cache = False

#config = LinearQuantizerConfig(module_type_configs={torch.nn.Linear: global_config})
config = LinearQuantizerConfig(module_type_configs={torch.nn.Linear: global_config})
quantizer = LinearQuantizer(StatefulQwen(model), config)

example_input_ids = torch.zeros((1, 1), dtype=torch.int32)
prepared_model = quantizer.prepare(example_inputs=(example_input_ids))

TraceError: symbolically traced variables cannot be used as inputs to control flow

In [ ]:
from torch.utils.data import DataLoader

# 1. Prepare a few strings that represent your app's actual use case
calibration_texts = [
"I feel like I’m falling behind everyone else and I don’t know how to catch up",
 "Today was exhausting and I can’t tell if I’m overreacting or just overwhelmed",
"I tried my best at work but it still feels like it wasn’t enough",
    "I keep replaying that awkward conversation in my head and cringing",
    "I miss how things used to be and I don’t know how to move forward", "I’m proud of myself for getting through today even though it was hard",
        "I feel lonely even when I’m around other people", "I snapped at someone I care about and now I feel guilty",
        "I don’t know why small things are making me so emotional lately", "I’m scared I’m making the wrong decision about my future",
            "I had a really good moment today and I want to hold onto it", "I feel stuck in a routine that’s draining me",
            "I wish I could explain how tired I am without sounding dramatic", "I’m worried I’m not living up to my potential", 
            "I avoided something important today and now it’s weighing on me", "I felt really anxious before that meeting and my heart wouldn’t slow down",
                "I’m trying to be kinder to myself but it’s not easy", "I feel like I’m carrying everyone else’s expectations", 
                "I had an argument and I don’t know how to fix it", "I keep comparing myself to others and it’s ruining my mood",
                "I did something brave today and I’m quietly proud of it", "I feel disconnected from my friends lately", "I’m afraid to tell people how much I’m struggling", 
                "I had a small win today that made me smile", "I feel guilty for resting when there’s so much to do", "I’m questioning whether I’m on the right path", 
                "I feel overwhelmed by responsibilities and don’t know where to start", "I’m trying to process something that hurt me", 
                "I want to change but I’m scared of failing again", "I just need someone to listen without judging me"

]

# 2. Tokenize them
calibration_data = []
for text in calibration_texts:
    # We use a context length that matches your target (e.g., 512 or 1024)
    tokens = tokenizer(
        text, 
        return_tensors="pt", 
        padding="max_length", 
        max_length=max_seq_len, 
        truncation=True
    )["input_ids"]
    calibration_data.append(tokens)

# 3. Create a simple DataLoader
calibration_loader = DataLoader(calibration_data, batch_size=1)

In [ ]:
prepared_model.eval()

with torch.no_grad():
   for data in calibration_loader:
        # 'data' is a tensor of shape (1, 512)
        # We only pass input_ids; the stateful wrapper's caches can be zeros here
        prepared_model(data)

In [ ]:
quantized_model = quantizer.finalize(inplace=True)
print("✅ W4A8 Quantization complete.")

In [ ]:
cache_shape = (1, num_heads, max_seq_len, head_dim)

# --- 2. DEFINE STATES ---
states = []
for i in range(num_layers):
    # We wrap a TensorType inside a StateType
    # This ensures KV caches are stored in FP16 to save memory
    states.append(ct.StateType(
        wrapped_type=ct.TensorType(
            shape=cache_shape, 
            dtype=np.float16
        ), 
        name=f"model.key_cache_{i}"
    ))
    states.append(ct.StateType(
        wrapped_type=ct.TensorType(
            shape=cache_shape, 
            dtype=np.float16
        ), 
        name=f"model.value_cache_{i}"
    ))
states.append(ct.StateType(
    wrapped_type=ct.TensorType(shape=(1,), dtype=np.int32), 
    name="model.step" 
))

In [3]:
from transformers.cache_utils import DynamicCache
import torch

class StatefulQwen(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, input_ids):
        # The model uses its internal buffers (registered above) 
        # as the 'past_key_values' automatically in recent transformers
        outputs = self.model(input_ids, use_cache=True)
        return outputs.logits

In [4]:
stateful_model = StatefulQwen(model) 

for i in range(num_layers):
    model.register_buffer(f"key_cache_{i}", torch.zeros(cache_shape, dtype=torch.float16))
    model.register_buffer(f"value_cache_{i}", torch.zeros(cache_shape, dtype=torch.float16))

# Also register a 'step' buffer to track the current token position
model.register_buffer("step", torch.zeros(1, dtype=torch.int32))

model = model.half()

In [5]:
print(model.model.rotary_emb.inv_freq.dtype)

torch.float16


In [6]:
example_input = torch.zeros((1, 1), dtype=torch.int32) # Single token trace

with torch.no_grad():
    scripted_model = torch.jit.script(stateful_model)

print(scripted_model.graph)

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


NotSupportedError: Compiled functions can't take variable number of arguments or use keyword-only arguments with defaults:
  File "/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/transformers/models/qwen3/configuration_qwen3.py", line 144
        layer_types: Optional[list[str]] = None,
        attention_dropout: Optional[float] = 0.0,
        **kwargs,
         ~~~~~~~ <--- HERE
    ):
        self.vocab_size = vocab_size


In [7]:
print("=== PARAMETERS ===")
for name, p in stateful_model.named_parameters():
    if p.dtype != torch.float16:
        print("❌", name, p.dtype)

print("=== BUFFERS ===")
for name, b in stateful_model.named_buffers():
    print(name, b.dtype)
    if name.startswith("key_cache") or name.startswith("value_cache"):
        assert b.dtype == torch.float16
    if name == "step":
        assert b.dtype == torch.int32

=== PARAMETERS ===
=== BUFFERS ===
model.key_cache_0 torch.float16
model.value_cache_0 torch.float16
model.key_cache_1 torch.float16
model.value_cache_1 torch.float16
model.key_cache_2 torch.float16
model.value_cache_2 torch.float16
model.key_cache_3 torch.float16
model.value_cache_3 torch.float16
model.key_cache_4 torch.float16
model.value_cache_4 torch.float16
model.key_cache_5 torch.float16
model.value_cache_5 torch.float16
model.key_cache_6 torch.float16
model.value_cache_6 torch.float16
model.key_cache_7 torch.float16
model.value_cache_7 torch.float16
model.key_cache_8 torch.float16
model.value_cache_8 torch.float16
model.key_cache_9 torch.float16
model.value_cache_9 torch.float16
model.key_cache_10 torch.float16
model.value_cache_10 torch.float16
model.key_cache_11 torch.float16
model.value_cache_11 torch.float16
model.key_cache_12 torch.float16
model.value_cache_12 torch.float16
model.key_cache_13 torch.float16
model.value_cache_13 torch.float16
model.key_cache_14 torch.float16


In [17]:
print(traced_model.graph)

with torch.no_grad():
    test_input = torch.zeros((1,1), dtype=torch.long)
    out = traced_model(test_input)

    if isinstance(out, tuple):
        out = out[0]

    print("Logits dtype:", out.dtype)

for name, tensor in traced_model.state_dict().items():
    if tensor.dtype == torch.float32:
        print("FLOAT32 FOUND:", name, tensor.dtype)

for node in traced_model.graph.nodes():
    node_str = str(node)
    if "Float" in node_str or "float" in node_str:
        print("Possible float32 node:")
        print(node_str)

for node in traced_model.graph.nodes():
    for output in node.outputs():
        print(output.debugName(), output.type())


graph(%self.1 : __torch__.StatefulQwen,
      %input_ids : Int(1, 1, strides=[1, 1], requires_grad=0, device=cpu)):
  %model : __torch__.transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM = prim::GetAttr[name="model"](%self.1)
  %14633 : Tensor = prim::CallMethod[name="forward"](%model, %input_ids)
  return (%14633)

Logits dtype: torch.float16
model __torch__.transformers.models.qwen3.modeling_qwen3.Qwen3ForCausalLM
14633 Tensor


In [ ]:
# Convert to a standard FP16 MLPackage
traced_model.eval()  # Ensure the model is in eval mode for conversion
mlmodel = ct.convert(
    traced_model,
    inputs=[ct.TensorType(name="input_ids", shape=(1, 1), dtype=np.int32)],
    states=states,
    outputs=[ct.TensorType(name="logits")],
    minimum_deployment_target=ct.target.iOS18,
    convert_to="mlprogram",
    compute_precision=ct.precision.FLOAT16
)

mlmodel.save("models/qwen3_1.7b_fp16.mlpackage")

Converting PyTorch Frontend ==> MIL Ops:   0%|          | 0/4123 [00:00<?, ? ops/s]Saving value type of int64 into a builtin type of int32, might lose precision!
Saving value type of int64 into a builtin type of int32, might lose precision!


ERROR - converting 'scaled_dot_product_attention' op (located at: 'model/model/0/self_attn/attn_output.1'):

Converting PyTorch Frontend ==> MIL Ops:   5%|▍         | 197/4123 [00:00<00:00, 8511.57 ops/s]


ValueError: In op, of type sub, named sub_1, the named input `y` must have the same data type as the named input `x`. However, y has dtype fp16 whereas x has dtype fp32.

In [ ]:


# For 4-bit weights specifically, we often use string identifiers or specific bit-widths
#config = LinearQuantizerConfig(
 #   global_config = global_config,
#)

In [14]:
# 1. Prepare dummy inputs as TUPLES
input_ids = torch.zeros((1, 1), dtype=torch.int32)
k_cache_tuple = tuple(torch.zeros(cache_shape) for _ in range(num_layers))
v_cache_tuple = tuple(torch.zeros(cache_shape) for _ in range(num_layers))

# 2. Trace with the nested tuple structure
traced_model = torch.jit.trace(
    StatefulQwen(quantized_model), 
    (input_ids, k_cache_tuple, v_cache_tuple)
)

NameError: name 'cache_shape' is not defined